# Lab 26-07 · Putting Numbers on Safety Claims

Companion to chapter 26-07. Write down each prediction before running the cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import safety as s

## Part 1 · Rare per unit, common per fleet

*Predict:* the fleet size at which a one-in-a-million failure has an even chance of happening at least once.

In [2]:
n50 = optimize.brentq(lambda n: s.p_any(1e-6, n) - 0.5, 1, 1e8)
print(f"fleet for a 50% chance: {n50:,.0f} units")
N = np.logspace(3, 10, 200)
fig, ax = plt.subplots(figsize=(5, 3.2))
for p in (1e-6, 1e-8):
    ax.semilogx(N, [s.p_any(p, n) for n in N], label=f"p = {p:g}")
ax.set_xlabel("units in the fleet"); ax.set_ylabel("P(at least one failure)"); ax.legend()

fleet for a 50% chance: 693,147 units


## Part 2 · Common cause

*Predict:* the beta factor at which common-cause and independent failures contribute equally, for p = 1e-3.

In [3]:
p = 1e-3
b_eq = optimize.brentq(lambda b: b * p - ((1 - b) * p) ** 2, 1e-6, 0.1)
print(f"equal contributions at beta = {b_eq:.2e}")
betas = np.logspace(-5, -1, 200)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.loglog(betas, [s.p_both_fail(p, b) for b in betas])
ax.axhline(p ** 2, ls="--", color="gray", label="independent")
ax.set_xlabel("beta (common-cause fraction)"); ax.set_ylabel("P(both channels fail)"); ax.legend()

equal contributions at beta = 9.98e-04


## Part 3 · Proving a rate

*Predict:* failure-free device-hours needed to show less than 1 FIT at 95% confidence.

In [4]:
print(f"hours for 1 FIT at 95%: {s.hours_to_demonstrate(1e-9, 0.95):.3g}")
fits = np.logspace(-1, 3, 100)
fig, ax = plt.subplots(figsize=(5, 3.2))
for conf in (0.90, 0.95):
    ax.loglog(fits, [s.hours_to_demonstrate(f * 1e-9, conf) for f in fits], label=f"{conf:.0%} confidence")
ax.set_xlabel("target rate (FIT)"); ax.set_ylabel("failure-free device-hours"); ax.legend()

hours for 1 FIT at 95%: 3e+09


## Part 4 · When failures occur

*Predict:* the 95% upper bound with 0, 1, 2 and 3 failures in 1e9 device-hours.

In [5]:
for k in range(4):
    print(f"{k} failures: upper bound {s.rate_upper_bound(1e9, k) / 1e-9:.2f} FIT")

0 failures: upper bound 3.00 FIT
1 failures: upper bound 4.74 FIT
2 failures: upper bound 6.30 FIT
3 failures: upper bound 7.75 FIT
